# Generative model constructing mock cluster samples

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scipy import interpolate
from scipy.integrate import cumulative_trapezoid
from scipy.stats.sampling import NumericalInversePolynomial
from scipy.stats import chi2
from tqdm import tqdm

from cloelib.cosmology import derived_cosmology
from cloelib.cosmology.camb_cosmology import (
    CAMBBackground,
    CAMBLinearPerturbations,
)
from cloelib.observables.clusters.halo_abundance import CastroHaloAbundance
from cloelib.observables.clusters.halo_mass_observable import (
    LognormalPowerLawHaloMassObservable,
)
from cloelib.observables.clusters.matter_statistics import MatterStatistics
from cloelib.observables.clusters.selection_function import GaussianSelectionFunction
from cloelib.summary_statistics.clusters import (
    ClusterCounts,
    ClusterStatisticsModeling,
)

### Mass and redshift ranges

True mass and redshift samples are drawn within these ranges. \
The halo mass function is normalised over these ranges.

In [ ]:
M_min = 5.e13  # M_sun / h
M_max = 2.e15  # M_sun / h
z_min = 0.2
z_max = 0.6

### Define the remaining model parameters and cosmology

In [ ]:
# Richness-mass scaling relation
alpha = 50.0
beta = 0.9
gamma = 1.

scatter0 = 0.1
scatterM = 0.0
scatterz = 0.0

M_piv = 1.e14
z_piv = 0.4

# P(lambda_ob|...)
sig_lambda_norm = 0.9
sig_lambda_z = 0.0
sig_lambda_exponent = 0.8

# P(z_ob|...)
sig_z_z = 0.1
sig_z_lambda = 0.0

# Define the survey area
area_degrees = 500 # deg^2
area = area_degrees * pow(np.pi / 180., 2)

# Cosmology
H0 = 70.0
h = H0 / 100.0
cosmo_pars = dict(
    H0=H0,
    Omega_cdm0=0.24,
    Omega_b0=0.022 / h**2,
    Omega_k0=0.0,
    w0=-1.0,
    wa=0.0,
    ns=0.96,
    mnu=0.06,
    As=2.1e-9,
    gamma_MG=0.0,
    N_mnu=1,
)

background = CAMBBackground(**cosmo_pars)
perturbations = CAMBLinearPerturbations(background, np.linspace(0.0, 2.0, 100))

### Build the halo models and the selection function

In [ ]:
matter_statistics = MatterStatistics(perturbations)

# Halo mass function
halo_abundance = CastroHaloAbundance(matter_statistics=matter_statistics)

# Halo mass density profile
#...

# Proxy-mass scaling relation
scaling_relation = LognormalPowerLawHaloMassObservable(
    alpha,
    beta,
    gamma,
    scatter0,
    scatterM,
    scatterz,
    M_piv,
    z_piv,
)

# Selection function
z_ob_edges = np.asarray([0.1, 0.3, 0.5, 0.8])
lambda_obs_edges = np.logspace(np.log10(20), np.log10(150), 10)

selection_function = GaussianSelectionFunction(
    scaling_relation,
    sig_lambda_norm,
    sig_lambda_z,
    sig_lambda_exponent,
    sig_z_z,
    sig_z_lambda,
    100,
    [100] * (len(lambda_obs_edges) - 1),
)

### Tabulate the joint mass–redshift density

Use $x=\ln M$ and include the Jacobian $dM=M\,dx$. Integrate the density with the trapezoidal rule, then invert the tabulated cumulatives with `np.interp`. This approximates each cumulative linearly between grid points. The conditional cumulative is interpolated over redshifts before inversion. Increase `n_z_grid` and `n_mass_grid` to check numerical convergence.


In [ ]:
def counts_in_volume_element(M, z, halo_abundance, background):
    z = np.atleast_1d(z)
    M = np.atleast_1d(M)
    dV = np.asarray(derived_cosmology.dV_dzdO(background, z, hubble_units=True))
    return dV[:, None] * halo_abundance.dn_dm(z, M)


n_z_grid = 250
n_mass_grid = 250
redshift_vector = np.linspace(z_min, z_max, n_z_grid)
log_mass_vector = np.linspace(np.log(M_min), np.log(M_max), n_mass_grid)
mass_vector = np.exp(log_mass_vector)

# Shape: (len(z), len(M))
joint_density = np.concatenate([
    counts_in_volume_element(mass_vector, zz, halo_abundance, background)
    * mass_vector[None, :]
    for zz in np.array_split(redshift_vector, 16)
])
if not np.all(np.isfinite(joint_density)) or np.any(joint_density < 0):
    raise ValueError("The joint density must be finite and nonnegative.")

# Unnormalized mass cumulatives at fixed z, integrated from ln(M_min).
# Shape: (n_z_grid, n_mass_grid)
mass_cumulative = cumulative_trapezoid(
    joint_density, x=log_mass_vector, axis=1, initial=0.0,
)

# Unnormalized redshift density: integrate over the full mass range.
# Shape: (n_z_grid)
Pz_samples = mass_cumulative[:, -1]  # get all z points, and the last logM point (which is the integral up to M_max)

# Unnormalized redshift cumulative, integrated from z_min.
# Shape: (n_z_grid,)
z_cumulative = cumulative_trapezoid(Pz_samples, x=redshift_vector, initial=0.0)

# Expected number of halos per steradian
n_halos = float(z_cumulative[-1])
if n_halos <= 0:
    raise ValueError("The expected abundance must be positive.")

# Expected number of halos in the survey area
N_halos = area * n_halos


def Pz_interp(z):
    return np.interp(z, redshift_vector, Pz_samples / n_halos, left=0.0, right=0.0)


print(f"Expected number of halos: {N_halos:.3f}")

### Generic PDF sampling

`scipy.stats.sampling.NumericalInversePolynomial` constructs an approximate inverse CDF from a PDF and normalizes it numerically.

Negative richness/redshift draws are mapped to zero.

In [ ]:
def sample_mass_redshift(size, rng):
    """
    Sample redshift, then mass conditional on redshift, from the global tables.

    Inputs: size (int), number of halos; rng (np.random.Generator), random source.
    Returns: mass in M_sun/h and redshift, each an array of shape (size,).
    """
    if np.any(np.diff(z_cumulative) <= 0) or np.any(np.diff(mass_cumulative, axis=1) <= 0): 
        raise ValueError("Inverse interpolation requires strictly increasing cumulatives.")

    z = np.interp(rng.random(size) * z_cumulative[-1], z_cumulative, redshift_vector)  # draw redshifts by inverting their cumulative
    mass_u = rng.random(size)  # draw uniform probabilities for conditional masses
    log_mass = np.empty(size)
    iz = np.clip(np.searchsorted(redshift_vector, z, side="right") - 1,  # find the lower redshift neighbour of each sample
                 0, len(redshift_vector) - 2)  # keep indices within valid intervals
    weights = (z - redshift_vector[iz]) / (redshift_vector[iz + 1] - redshift_vector[iz])  # fractional position between redshift neighbours

    for j, (row, weight) in enumerate(zip(iz, weights)):  # process each sampled redshift
        cumulative = (1 - weight) * mass_cumulative[row] + weight * mass_cumulative[row + 1]  # interpolate its conditional mass cumulative
        log_mass[j] = np.interp(mass_u[j] * cumulative[-1], cumulative, log_mass_vector)  # invert the cumulative to draw ln(M)
    return np.exp(log_mass), z  # convert ln(M) to mass and return both arrays


class ScaledPDF:
    """
    Rescale a PDF to ease numerical sampling.

    The aim of this function is to sample values close to zero, for numerical performance.
    For the original variable y, use x = (y - location) / scale, where
    location is the mean of the distribution and scale is its width.
    The density becomes p_x(x) = scale * p_y(location + scale * x).
    The factor scale is the Jacobian dy/dx and preserves probability.

    Inputs: pdf (callable); location; scale; support (pair), i.e. bounds.
    Output: a pdf(x) for SciPy, returning the transformed density within
    the support. No draws are made here.
    """
    def __init__(self, pdf, location, scale, support):
        if not np.isfinite(location) or not np.isfinite(scale) or scale <= 0:
            raise ValueError("PDF location must be finite and scale strictly positive.")
        self.function = pdf
        self.location = float(location)
        self.scale = float(scale)
        self.support_bounds = support

    def pdf(self, x):
        value = self.location + self.scale * x
        low, high = self.support_bounds
        if value <= low or value >= high:
            return 0.0
        result = float(np.asarray(self.function(value)).item()) * self.scale
        if not np.isfinite(result) or result < 0:
            raise ValueError("PDF must be finite and nonnegative inside its support.")
        return result


def draw_pdf(pdf, rng, *, location, scale, support=(-np.inf, np.inf), u_resolution=1e-8):
    """
    Draw one value from a PDF by numerical inverse-CDF sampling.

    Use ScaledPDF to work in x = (y - location) / scale and transform the
    support to the same coordinates. SciPy builds the inverse CDF from this
    density, normalizing it numerically and handling infinite tails.
    Convert the draw back with y = location + scale*x.

    Inputs: pdf (callable), density in the original variable y;
    rng (np.random.Generator), random source; location and scale, numerical
    centering and width (scale must be positive); support (pair),
    bounds in y; u_resolution (float), target inverse-CDF probability error.
    Returns: one float in the original coordinates.
    """
    dist = ScaledPDF(pdf, location, scale, support)
    domain = tuple((bound - location) / scale for bound in support)
    sampler = NumericalInversePolynomial(
        dist, domain=domain, center=0.0, u_resolution=u_resolution, random_state=rng,
    )
    return float(location + scale * sampler.rvs())


def draw_true_richness(z, mass, rng):
    location = float(np.exp(scaling_relation._mean_lnrichness(z, mass)))
    scale = location * float(scaling_relation.scatter_lnrichness(z, mass))
    return draw_pdf(lambda value: scaling_relation._pdf_richness(z, mass, value),
                    rng, location=location, scale=scale, support=(0.0, np.inf))


def draw_observed_richness(z, richness, rng):
    return draw_pdf(lambda value: selection_function._prob_lambda_obs(z, richness, value),
                    rng, location=richness,
                    scale=float(selection_function._scatter_lambda_obs(z, richness)))


def draw_observed_redshift(z, richness_obs, rng):
    return draw_pdf(lambda value: selection_function._prob_z_obs(value, richness_obs, z),
                    rng, location=z,
                    scale=float(selection_function.scatter_z_obs(richness_obs, z)))

### Draw a reproducible Poisson catalogue

Set `poisson=False` for a fixed count `int(expected_count)`. No global random state or worker processes are used.

In [ ]:
SEED = 42


def get_samples(expected_count=N_halos, seed=SEED, poisson=True, progress=True):
    rng = np.random.default_rng(seed)
    N = int(rng.poisson(expected_count)) if poisson else int(expected_count)
    mass, redshift = sample_mass_redshift(N, rng)
    samples = np.empty((N, 5))
    for i in tqdm(range(N), disable=not progress, desc="Sampling"):
        M, z = mass[i], redshift[i]
        lambda_tr = max(0.0, draw_true_richness(z, M, rng))
        lambda_ob = max(0.0, draw_observed_richness(z, lambda_tr, rng))
        z_ob = max(0.0, draw_observed_redshift(z, lambda_ob, rng))
        samples[i] = M, z, lambda_tr, lambda_ob, z_ob
    return pd.DataFrame(samples, columns=["M", "z", "lambda", "lambda_ob", "z_ob"])


df = get_samples()
print(f"Expected: {N_halos:.3f}; drawn: {len(df)}; seed: {SEED}\n")

df

### Diagnostics plots

Compare sampled histograms with the expected values from cloelib.

In [ ]:
from scipy.integrate import quad_vec, simpson
from matplotlib.ticker import FuncFormatter, LogLocator

# Independent integration grids for diagnostics
DIAG_NZ = 65
DIAG_NM = 129
DIAG_NL = 801


def diagnostic_model(z_bounds=(z_min, z_max), mass_bounds=(M_min, M_max)):
    """Build cloelib integrators on the given intervals."""
    return ClusterStatisticsModeling(
        halo_abundance=halo_abundance,
        selection_function=selection_function,
        integ_k_arr=np.geomspace(2e-4, 9, 500),
        integ_mass_arr=np.geomspace(*mass_bounds, DIAG_NM),
        integ_lambda_true_arr=np.geomspace(0.1, 1.e4, DIAG_NL),
        integ_ztrue_arr=np.linspace(*z_bounds, DIAG_NZ),
        area=area_degrees,
    )


def integrate_z(model, values):
    """Integrate values shaped (z, ...) with cloelib's survey-volume weight."""
    nz = model.tabulated_integrands['ztrue'].size
    return model.integrate_probe_function_in_redshift(
        np.asarray(values)[None, ...], np.ones((1, 1, nz)),
    )[0, 0]


def integrate_population(model, kernel=None):
    """Integrate a (z, M, ...) kernel with cloelib's abundance and volume weights."""
    nz = model.tabulated_integrands['ztrue'].size
    nm = model.tabulated_integrands['M'].size
    if kernel is None:
        kernel = np.ones((nz, nm))
    mass_integral = model.integrate_probe_function_in_mass(
        kernel, np.ones((1, nz, nm)),
    )[0]
    return integrate_z(model, mass_integral)


def pdf_bin_probabilities(pdf, edges, clipped_at_zero=False):
    """Integrate a PDF over bins; its last output axis indexes x."""
    widths = np.diff(edges)
    probabilities = quad_vec(
        lambda t: pdf(edges[:-1] + t * widths) * widths,
        0.0, 1.0, epsabs=1.e-9, epsrel=1.e-7,
    )[0]
    if clipped_at_zero:
        if edges[0] != 0:
            raise ValueError('Clipped-observable bins must start at zero.')
        probabilities[..., 0] += quad_vec(
            lambda x: np.asarray(pdf(np.array([x])))[..., 0],
            -np.inf, 0.0, epsabs=1.e-9, epsrel=1.e-7,
        )[0]
    return probabilities


def plot_integrated_pdf(values, edges, probabilities, xlabel, ylabel, label,
                        logx=False, logy=False):
    """Compare densities per bin, keeping normalization outside the plotted range."""
    if len(values) == 0:
        raise ValueError('No sampled objects in the conditioning interval.')
    fig, ax = plt.subplots()
    ax.set_xlabel(xlabel, fontsize=13)
    ax.set_ylabel(ylabel, fontsize=13)
    counts = np.histogram(values, bins=edges)[0]
    widths = np.diff(edges)
    ax.stairs(counts / (len(values) * widths), edges, label=label, baseline=None)
    ax.stairs(probabilities / widths, edges, color='r', label='Integrated cloelib model', baseline=None)
    if logx:
        ax.set_xscale('log')
    if logy:
        ax.set_yscale('log')
        # Show densities resolvable by this sample, rather than arbitrarily tiny tails
        sample_density = counts / (len(values) * widths)
        positive = sample_density[sample_density > 0]
        if positive.size:
            ax.set_ylim(positive.min() / 5, 2 * max(sample_density.max(), (probabilities / widths).max()))
    ax.legend(fontsize=10)
    plt.show()


# P(z): marginalize over all masses, then integrate over each redshift bin
z_edges_check = np.linspace(z_min, z_max, 50)
population_model = diagnostic_model()
population_norm = integrate_population(population_model)
z_probabilities = np.array([
    integrate_population(diagnostic_model(z_bounds=(low, high)))
    for low, high in zip(z_edges_check[:-1], z_edges_check[1:])
]) / population_norm
plot_integrated_pdf(df['z'], z_edges_check, z_probabilities, '$z$', '$P(z)$', 'Samples')

In [ ]:
# P(M | 0.3 < z < 0.4): integrate over the selected redshift interval.
z_slice = (0.3, 0.4)
df_test = df[(df['z'] > z_slice[0]) & (df['z'] < z_slice[1])]
mass_edges_check = np.geomspace(M_min, M_max, 41)
mass_slice_norm = integrate_population(diagnostic_model(z_bounds=z_slice))
mass_probabilities = np.array([
    integrate_population(diagnostic_model(z_bounds=z_slice, mass_bounds=(low, high)))
    for low, high in zip(mass_edges_check[:-1], mass_edges_check[1:])
]) / mass_slice_norm
plot_integrated_pdf(df_test['M'], mass_edges_check, mass_probabilities,
                    r'$M [\mathrm{M}_\odot/h]$', r'$P(M|\Delta z)$', r'Samples, $z\in[0.3,0.4]$', logx=True, logy=True)

In [ ]:
# P(lambda | mass and redshift ranges): abundance-weighted mixture of model PDFs.
z_slice = (0.3, 0.4)
mass_slice = (1.e14, 2.e14)
df_test = df[
    (df['M'] > mass_slice[0]) & (df['M'] < mass_slice[1])
    & (df['z'] > z_slice[0]) & (df['z'] < z_slice[1])
]
richness_model = diagnostic_model(z_bounds=z_slice, mass_bounds=mass_slice)
z_tab = richness_model.tabulated_integrands['ztrue']
mass_tab = richness_model.tabulated_integrands['M']
richness_edges_check = np.geomspace(5.0, 300.0, 41)
richness_kernel = pdf_bin_probabilities(
    lambda x: scaling_relation.pdf_richness(z_tab, mass_tab, x),
    richness_edges_check,
)
richness_probabilities = integrate_population(richness_model, richness_kernel) / integrate_population(richness_model)
plot_integrated_pdf(df_test['lambda'], richness_edges_check, richness_probabilities,
                r'$\lambda$', r'$P(\lambda|\Delta M, \Delta z)$', 'Samples,\n'+r'$M\in[1,2]\times10^{14}$, $z\in[0.3,0.4]$',
                    logx=True, logy=True)

In [ ]:
integ_k_arr = np.geomspace(2e-4, 9, 500)

#integ_mass_arr = np.logspace(12.0, 16.0, 51)
integ_mass_arr = np.geomspace(M_min, M_max, 101)

integ_lambda_true_arr = np.geomspace(0.1, 1.e4, 401)  # Check convergence for new PDF models.
integ_ztrue_arr = np.linspace(z_min, z_max, 81)

#
cluster_statistics_modeling = ClusterStatisticsModeling(
    halo_abundance,
    selection_function,
    integ_k_arr,
    integ_mass_arr,
    integ_lambda_true_arr,
    integ_ztrue_arr,
    area=area_degrees
)

# z window
window = cluster_statistics_modeling.window_z_observed(
    z_ob_edges, lambda_obs_edges
)

# ----

def check_likelihood_model(test_measure, test_model):
    assert np.all(np.isfinite(test_model))
    assert np.all(test_model > 0)
    chi2_model = np.sum((test_measure - test_model)**2 / test_model)
    assert chi2.sf(chi2_model, test_model.size) > 0.01

In [ ]:
# testing counts
CC = ClusterCounts(cluster_statistics_modeling, None)
counts_model = CC.get_NC(z_ob_edges, lambda_obs_edges, False)

counts_measure = np.zeros((len(z_ob_edges)-1, len(lambda_obs_edges)-1))
for i in range(len(z_ob_edges)-1):
    for j in range(len(lambda_obs_edges)-1):
        counts_measure[i][j] = len(df[
            (df['z_ob'] >= z_ob_edges[i]) &
            (df['z_ob'] < z_ob_edges[i+1]) &
            (df['lambda_ob'] >= lambda_obs_edges[j]) &
            (df['lambda_ob'] < lambda_obs_edges[j+1])
        ])

check_likelihood_model(counts_measure, counts_model)

#
lambda_ob_bin_centers = 0.5 * (lambda_obs_edges[:-1] + lambda_obs_edges[1:])
print(lambda_ob_bin_centers)

#
fig, ax = plt.subplots(1, 3, figsize=(14,4.5), sharex='all', sharey='all', gridspec_kw={'wspace':0})
for axis in ax:
    axis.tick_params(which='both', direction='in', labelsize=12)
    axis.set_xlabel(r"$\lambda_{\rm ob}$", fontsize=14)
    axis.set_xscale('log')
    axis.set_yscale('log')
    axis.set_xlim(15, 150)
    # Keep logarithmic scales, but display ordinary decimal tick labels.
    for coordinate_axis in (axis.xaxis, axis.yaxis):
        coordinate_axis.set_major_formatter(FuncFormatter(lambda value, pos: np.format_float_positional(value, trim='-')))
        coordinate_axis.set_minor_formatter(FuncFormatter(lambda value, pos: np.format_float_positional(value, trim='-')))
    axis.xaxis.set_major_locator(LogLocator(base=10))
    axis.xaxis.set_minor_locator(LogLocator(base=10, subs=(2, 5)))
    axis.yaxis.set_minor_locator(LogLocator(base=10, subs=(2, 5)))
ax[0].set_ylabel(r"$N\,(\Delta\lambda_{\rm ob}, \Delta z_{\rm ob})$", fontsize=14)

for i in range(len(z_ob_edges)-1):
    ax[i].errorbar(lambda_ob_bin_centers, counts_measure[i], np.sqrt(counts_measure[i]), color='k', capsize=4, markersize=4, fmt='o')
    ax[i].plot(lambda_ob_bin_centers, counts_model[i], color='darkorange')

plt.show()